# 03. Avaliacao, relatorio e demo

Notebook canonico para executar a fase final do pipeline: Whisper, metricas automaticas, agregacao, geracao de `report_assets/` e inicializacao da demo Gradio.

Sobrescritas usuais ficam concentradas na celula de parametros: `NISQA_PATH` quando o checkout nao estiver em `./NISQA` e `DEMO_HOST` ou `DEMO_PORT` quando a exposicao da demo exigir outro binding.


In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    markers = ('pyproject.toml', 'README.md', 'scripts')
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    workspace = Path('/workspace')
    if workspace.exists():
        candidates.append(workspace.resolve())
        for child in sorted(workspace.iterdir()):
            if child.is_dir():
                candidates.append(child.resolve())
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto a partir do notebook.')


PROJECT_ROOT = find_project_root()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'


def resolve_path(path: str | Path) -> Path:
    candidate = Path(path)
    return candidate if candidate.is_absolute() else PROJECT_ROOT / candidate


def assert_path(path: str | Path, *, kind: str | None = None) -> Path:
    resolved = resolve_path(path)
    if not resolved.exists():
        raise FileNotFoundError(f'Path not found: {resolved}')
    if kind == 'file' and not resolved.is_file():
        raise FileNotFoundError(f'Expected file, found: {resolved}')
    if kind == 'dir' and not resolved.is_dir():
        raise FileNotFoundError(f'Expected directory, found: {resolved}')
    print(f'OK: {resolved}')
    return resolved


def run_cmd(args: list[object], env: dict[str, object] | None = None, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [str(arg) for arg in args]
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items() if value is not None})
    print('+', shlex.join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, env=merged_env, check=check, text=True)


def run_project_python(args: list[object], env: dict[str, object] | None = None) -> subprocess.CompletedProcess:
    assert_path(VENV_PYTHON, kind='file')
    return run_cmd([VENV_PYTHON, *args], env=env)


def preview_csv(path: str | Path, rows: int = 5, columns: list[str] | None = None) -> pd.DataFrame:
    resolved = assert_path(path, kind='file')
    frame = pd.read_csv(resolved)
    if columns:
        frame = frame.loc[:, columns]
    print(f'rows={len(frame)} columns={list(frame.columns)}')
    display(frame.head(rows))
    return frame


print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'VENV_PYTHON={VENV_PYTHON}')


## Parametros

In [ ]:
CONFIG_PATH = Path('configs/speecht5_minimal.yaml')
SAMPLES_PATH = Path('artifacts/evaluation/samples.csv')
EVALUATION_DIR = Path('artifacts/evaluation')
METRICS_PATH = EVALUATION_DIR / 'metrics_summary.csv'
COSTS_PATH = EVALUATION_DIR / 'cost_summary.csv'
REPORT_ASSETS_DIR = Path('report_assets')
NISQA_PATH = PROJECT_ROOT / 'NISQA'
DEMO_HOST = '0.0.0.0'
DEMO_PORT = 7860


## Validacoes de pre-requisito

In [ ]:
assert_path(VENV_PYTHON, kind='file')
assert_path(CONFIG_PATH, kind='file')
assert_path(SAMPLES_PATH, kind='file')

samples_pre = pd.read_csv(resolve_path(SAMPLES_PATH))
audio_paths = samples_pre['audio_path'].fillna('').astype(str).str.strip()
materialized_audio = [resolve_path(path) for path in audio_paths if path]
if not materialized_audio:
    raise RuntimeError('Nenhum audio materializado encontrado em samples.csv. Rode o notebook 02 antes deste.')
existing_audio = sum(path.exists() for path in materialized_audio)
print(f'audio_rows={len(materialized_audio)} existing_audio_files={existing_audio}')
preview_csv(SAMPLES_PATH, columns=['sample_id', 'condition', 'checkpoint_label', 'speaker_id', 'status', 'audio_path'])


## 1. Transcrever audios com Whisper

In [ ]:
run_project_python([
    'scripts/run_whisper_batch.py',
])


## 2. Calcular WER

In [ ]:
run_project_python([
    'scripts/compute_wer.py',
])


## 3. Calcular speaker similarity

In [ ]:
run_project_python([
    'scripts/compute_speaker_similarity.py',
])


## 4. Calcular NISQA

In [ ]:
assert_path(NISQA_PATH, kind='dir')
command: list[object] = [
    'scripts/compute_nisqa.py',
]
if NISQA_PATH != PROJECT_ROOT / 'NISQA':
    command.extend(['--nisqa-path', NISQA_PATH])
run_project_python(command)


## 5. Calcular F0 RMSE

In [ ]:
run_project_python([
    'scripts/compute_f0_rmse.py',
])


## 6. Agregar resultados

In [ ]:
run_project_python([
    'scripts/aggregate_metrics.py',
])

assert_path(METRICS_PATH, kind='file')
assert_path(COSTS_PATH, kind='file')


## 7. Gerar report_assets

In [ ]:
run_project_python([
    'scripts/make_report_assets.py',
])

assert_path(REPORT_ASSETS_DIR, kind='dir')


## Inspecoes tabulares finais

In [ ]:
preview_csv(SAMPLES_PATH, columns=['sample_id', 'checkpoint_label', 'speaker_id', 'prompt_id', 'status', 'wer', 'speaker_similarity', 'nisqa', 'f0_rmse'])
preview_csv(METRICS_PATH)
preview_csv(COSTS_PATH)


## Demo Gradio

A celula abaixo sobe a demo em `0.0.0.0:7860`. No RunPod/Jupyter, exponha a porta correspondente no painel do pod para abrir a interface no navegador.

In [ ]:
command: list[object] = [
    'demo/app.py',
]
if DEMO_HOST != '127.0.0.1':
    command.extend(['--host', DEMO_HOST])
if DEMO_PORT != 7860:
    command.extend(['--port', DEMO_PORT])
run_project_python(command)
